# OSFI Regulation Assistant: a RAG project
## Part 1: Collect, clean, and chunk the documents

**What we're building.** An assistant that answers questions about Canadian mortgage and model-risk regulation, using only official OSFI guidelines, with citations to the exact section each answer came from.

**What is RAG?** *Retrieval-Augmented Generation.* An LLM on its own answers from memory, which can be outdated or invented. A RAG system first **retrieves** the most relevant passages from trusted documents, then asks the LLM to **generate** an answer using only those passages. It's the standard way companies build assistants over their own documents.

**The roadmap:**
1. **Part 1 (this notebook):** download the documents, extract clean text, and split it into chunks
2. **Part 2:** turn chunks into embeddings and build vector search
3. **Part 3:** generate grounded answers with an LLM, with citations
4. **Part 4:** evaluate retrieval and answer quality with a test set
5. **Part 5:** serve it as an API with FastAPI, package it with Docker, and publish it

**How to run:** the default CPU runtime is fine for Part 1. Run cells in order with **Shift + Enter**.

## Step 1: Imports and project folder on Google Drive

In [ ]:
import os, re, json, time
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/osfi_rag_project"
except Exception as e:
    print("Drive not available, using a local folder:", e)
    PROJECT_DIR = "osfi_rag_project"

RAW_DIR = os.path.join(PROJECT_DIR, "raw_html")
DATA_DIR = os.path.join(PROJECT_DIR, "data")
os.makedirs(RAW_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## Step 2: Choose the documents

Each document gets a short **ID**, used later in citations (for example `B-20`), a readable title, and its official URL. Keeping this list in one place makes it easy to add documents later.

*Source: Office of the Superintendent of Financial Institutions (OSFI). Only the URLs are stored in this project, and the text is downloaded when you run the notebook.*

In [ ]:
BASE = "https://www.osfi-bsif.gc.ca/en/guidance/guidance-library/"
DOCUMENTS = [
    {"doc_id": "B-20",
     "title": "Guideline B-20: Residential Mortgage Underwriting Practices and Procedures (2017)",
     "url": BASE + "residential-mortgage-underwriting-practices-procedures-guideline-2017"},
    {"doc_id": "B-21",
     "title": "Guideline B-21: Residential Mortgage Insurance Underwriting Practices and Procedures (2019)",
     "url": BASE + "residential-mortgage-insurance-underwriting-practices-procedures-guideline-2019"},
    {"doc_id": "B-20-INFO",
     "title": "Infosheet: Residential Mortgage Underwriting Practices and Procedures Guideline (B-20)",
     "url": BASE + "infosheet-residential-mortgage-underwriting-practices-procedures-guideline-b-20"},
    {"doc_id": "B-20-CLAR",
     "title": "Clarification on the Treatment of Innovative Real Estate Secured Lending Products under Guideline B-20",
     "url": BASE + "clarification-treatment-innovative-real-estate-secured-lending-products-under-guideline-b-20"},
    {"doc_id": "MI-NOTE",
     "title": "Implementation Note on Mortgage Insurance",
     "url": BASE + "mortgage-insurance"},
    {"doc_id": "E-23",
     "title": "Guideline E-23: Model Risk Management (2027)",
     "url": BASE + "guideline-e-23-model-risk-management-2027"},
]
docs_df = pd.DataFrame(DOCUMENTS)
docs_df

## Step 3: Download the pages

We save each page's raw HTML to Drive, so we only download once. Being polite to the website matters: we wait between requests and identify our script with a *User-Agent* header.

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (educational RAG portfolio project)"}

def download(doc):
    path = os.path.join(RAW_DIR, f"{doc['doc_id']}.html")
    if os.path.exists(path):
        return path, "cached"
    r = requests.get(doc["url"], headers=HEADERS, timeout=60)
    r.raise_for_status()
    with open(path, "w", encoding="utf-8") as f:
        f.write(r.text)
    time.sleep(2)   # pause between requests
    return path, f"downloaded ({len(r.text) / 1024:.0f} KB)"

for doc in DOCUMENTS:
    try:
        path, status = download(doc)
        print(f"{doc['doc_id']:<10} {status}")
    except Exception as e:
        print(f"{doc['doc_id']:<10} FAILED: {e}")

## Step 4: Extract the text, keeping the document structure

A web page contains much more than the guideline: menus, a search box, a feedback form, and links. We keep only the **main content** and walk through it in order, recording:

- **Headings** (`h2`, `h3`, `h4`), so every passage knows which section it belongs to. This becomes the citation later, for example *B-20 › Principle 3 › Debt service coverage*.
- **Paragraphs, list items, and table rows**, the actual content.

We skip the table of contents and other link-only lists, bold labels like "Information", page metadata above the first heading, the feedback form, and footnote "return" links.

**Two clean-ups based on the first run:**
- **Footnotes.** OSFI places footnote text inside sentences with a custom `<fn>` tag, which garbled sentences like *"...institutions (FRFIs For the purposes of this Implementation Note...)"*. We move each footnote out of its sentence and keep it as a separate passage in a **Footnotes** section, since footnotes often hold useful definitions.
- **Chart data.** The B-20 infosheet's charts include data tables, producing hundreds of rows like *"Jul-15 | 25% | 22%"*. Rows with fewer than four real words are skipped, along with "Source:" lines, so search isn't cluttered with numbers.

In [ ]:
SKIP_SELECTORS = ["script", "style", "nav", "form", "header", "footer",
                  ".pagedetails", "#wb-dtmd", "section.pagedetails", ".gc-pgdtls"]
STOP_PHRASES = ("Report a problem or mistake on this page", "Date modified")

def clean_text(s):
    s = re.sub(r"Return to footnote\s*\d+\s*referrer", " ", s)
    s = re.sub(r"\bFootnote\s*\d+\b", " ", s)          # inline footnote markers
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s)
    return re.sub(r"\s+([),.;:])", r"\1", s).strip()   # no space before closing punctuation

def is_link_only(el):
    """True for list items that are just a link (table of contents, related documents)."""
    links = el.find_all("a")
    return len(links) == 1 and clean_text(links[0].get_text()) == clean_text(el.get_text())

def is_bold_label(el, text):
    """True for short paragraphs that are only a bold label, like "Information" or "Table of contents"."""
    strong = el.find(["strong", "b"])
    return strong is not None and len(text.split()) <= 4 and clean_text(strong.get_text(" ")) == text

def is_number_row(text):
    """True for chart data rows like 'Jul-15 | 25% | 22%': fewer than 4 real words."""
    return len(re.findall(r"\b[A-Za-z]{3,}\b", text)) < 4

SKIPPED = {"number_rows": 0, "source_lines": 0, "footnotes_moved": 0}

def extract_blocks(html, doc_id):
    soup = BeautifulSoup(html, "html.parser")
    main = soup.find("main") or soup.body
    for sel in SKIP_SELECTORS:
        for el in main.select(sel):
            el.decompose()
    # OSFI puts footnote text inside sentences with a custom <fn> tag.
    # Move each footnote out of the sentence and keep it as its own passage.
    footnotes = []
    for fn in main.find_all("fn"):
        fn_text = clean_text(fn.get_text(" "))
        if fn_text:
            footnotes.append(f"Footnote {fn.get('value', '')}: {fn_text}".replace("Footnote :", "Footnote:"))
        fn.decompose()
    SKIPPED["footnotes_moved"] += len(footnotes)

    title_el = main.find("h1")
    page_title = clean_text(title_el.get_text()) if title_el else doc_id

    blocks, path = [], {}
    for el in main.find_all(["h2", "h3", "h4", "h5", "p", "li", "tr"]):
        text = clean_text(el.get_text(" "))
        if not text:
            continue
        if text.startswith(STOP_PHRASES):
            break
        name = el.name
        if name in ("h2", "h3", "h4", "h5"):
            level = int(name[1])
            path = {k: v for k, v in path.items() if k < level}
            path[level] = text
            continue
        # avoid counting the same text twice (a paragraph inside a list item or table row)
        if name in ("p", "li") and el.find_parent(["li", "tr"]):
            continue
        if name in ("p", "li") and is_link_only(el):          # table of contents, "Generate PDF" etc.
            continue
        if name == "p" and is_bold_label(el, text):            # labels like "Information", "Table of contents"
            continue
        if not path and (name in ("li", "tr") or len(text.split()) < 8):
            continue                                           # page metadata above the first heading
        if name == "tr":
            cells = [clean_text(c.get_text(" ")) for c in el.find_all(["th", "td"])]
            text = " | ".join(c for c in cells if c)
            if is_number_row(text):                            # chart data, not readable text
                SKIPPED["number_rows"] += 1
                continue
        if text.startswith("Source:"):                         # chart source notes
            SKIPPED["source_lines"] += 1
            continue
        section = " › ".join(path[k] for k in sorted(path)) or "Introduction"
        blocks.append({"doc_id": doc_id, "page_title": page_title, "section": section,
                       "kind": name, "text": text})
    for t in footnotes:
        blocks.append({"doc_id": doc_id, "page_title": page_title, "section": "Footnotes",
                       "kind": "footnote", "text": t})
    return blocks

all_blocks = []
for doc in DOCUMENTS:
    path = os.path.join(RAW_DIR, f"{doc['doc_id']}.html")
    if not os.path.exists(path):
        print("Missing:", doc["doc_id"]); continue
    with open(path, encoding="utf-8") as f:
        all_blocks += extract_blocks(f.read(), doc["doc_id"])

blocks = pd.DataFrame(all_blocks)
before = len(blocks)
blocks = blocks.drop_duplicates(["doc_id", "text"]).reset_index(drop=True)   # repeated passages
blocks["words"] = blocks["text"].str.split().str.len()
print(f"{len(blocks):,} text blocks extracted")
print(f"Cleaned up: {SKIPPED['footnotes_moved']} footnotes moved out of sentences, "
      f"{SKIPPED['number_rows']} chart-data rows and {SKIPPED['source_lines']} source lines skipped, "
      f"{before - len(blocks)} duplicate passages removed")
blocks.head(10)

## Step 5: Check the extraction quality

Just like the 311 project, we check the data before trusting it:
- Did every document produce text?
- Are the section names sensible?
- Is there leftover website noise (menus, buttons, "Share this page")?

In [ ]:
summary = (blocks.groupby("doc_id")
           .agg(blocks=("text", "size"), words=("words", "sum"), sections=("section", "nunique"))
           .reindex([d["doc_id"] for d in DOCUMENTS]))
summary

In [ ]:
# The section headings found in each document (first 12 per document)
for doc_id, g in blocks.groupby("doc_id", sort=False):
    print(f"\n=== {doc_id} ===")
    for s in g["section"].drop_duplicates().head(12):
        print("  ", s)

In [ ]:
# Very short blocks are often website leftovers. Look at them to decide whether to filter anything.
blocks[blocks["words"] <= 4][["doc_id", "section", "text"]].drop_duplicates("text").head(25)

In [ ]:
# Exact duplicate passages (repeated boilerplate shows up here)
dupes = blocks[blocks.duplicated("text", keep=False)]
print(f"{dupes['text'].nunique()} passages appear more than once")
dupes[["doc_id", "section", "text"]].drop_duplicates("text").head(10)

## Step 6: Split the text into chunks

**Why chunk?** An LLM can only read a limited amount of text per question, and search works best on focused passages. So we split documents into chunks of roughly **200 words**.

**How we chunk (and why):**
- **Chunks never cross a section boundary.** A chunk about *debt service coverage* shouldn't drift into *amortization*. This keeps each chunk about one topic and makes citations precise.
- **We split on sentences**, never mid-sentence.
- **Neighbouring chunks overlap by one sentence**, so an idea split across two chunks isn't lost.
- **Each chunk gets a short header** naming its document and section. This gives the search step context that a bare paragraph lacks, a technique called *contextual chunk headers*.

In [ ]:
TARGET_WORDS = 200     # aim for about this many words per chunk
OVERLAP_SENTENCES = 1  # sentences repeated at the start of the next chunk

SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z(“\"])")

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT.split(text) if s.strip()]

titles = {d["doc_id"]: d["title"] for d in DOCUMENTS}
urls = {d["doc_id"]: d["url"] for d in DOCUMENTS}

chunks = []
for (doc_id, section), g in blocks.groupby(["doc_id", "section"], sort=False):
    sentences = []
    for text in g["text"]:
        sentences += split_sentences(text)
    current, count = [], 0
    for sent in sentences:
        n = len(sent.split())
        if current and count + n > TARGET_WORDS:
            chunks.append((doc_id, section, " ".join(current)))
            current = current[-OVERLAP_SENTENCES:] if OVERLAP_SENTENCES else []
            count = sum(len(s.split()) for s in current)
        current.append(sent); count += n
    if current:
        chunks.append((doc_id, section, " ".join(current)))

chunks_df = pd.DataFrame(chunks, columns=["doc_id", "section", "text"])
chunks_df["chunk_id"] = chunks_df["doc_id"] + "-" + (chunks_df.groupby("doc_id").cumcount() + 1).astype(str).str.zfill(3)
chunks_df["title"] = chunks_df["doc_id"].map(titles)
chunks_df["url"] = chunks_df["doc_id"].map(urls)
chunks_df["words"] = chunks_df["text"].str.split().str.len()
chunks_df["embed_text"] = ("Document: " + chunks_df["title"] + "\nSection: " + chunks_df["section"]
                           + "\n\n" + chunks_df["text"])
chunks_df = chunks_df[["chunk_id", "doc_id", "title", "section", "url", "words", "text", "embed_text"]]
print(f"{len(chunks_df):,} chunks")
chunks_df.groupby("doc_id", sort=False)["words"].agg(["count", "mean", "min", "max"]).round(0)

### Chunk size distribution
Most chunks should sit near the 200-word target. Very small chunks come from short sections (for example, a heading with a single sentence), which is fine.

In [ ]:
import matplotlib.pyplot as plt
ax = chunks_df["words"].plot(kind="hist", bins=30, figsize=(8, 3), color="#4c72b0")
ax.set_title("Words per chunk"); ax.set_xlabel("Words"); plt.tight_layout(); plt.show()

### Look at a few chunks, exactly as the search step will see them

In [ ]:
for _, row in chunks_df.sample(3, random_state=7).iterrows():
    print("=" * 100)
    print(row["chunk_id"], f"({row['words']} words)")
    print(row["embed_text"][:1200])

## Step 7: Save the chunks

We save the chunks to Drive as **JSON Lines** (one chunk per line), a simple format that Part 2 will load directly.

In [ ]:
out_path = os.path.join(DATA_DIR, "chunks.jsonl")
chunks_df.to_json(out_path, orient="records", lines=True, force_ascii=False)
docs_df.to_csv(os.path.join(DATA_DIR, "documents.csv"), index=False)
print("Saved", len(chunks_df), "chunks to", out_path)

## Checkpoint

**Send me the outputs of:**
1. **Step 3:** the download messages
2. **Step 5:** the summary table, the section headings, and the short-blocks table
3. **Step 6:** the chunk statistics table, plus one of the sample chunks from the last cell of Step 6

I couldn't test against OSFI's live pages from my side, so this check matters. If any website noise slipped through, we'll filter it before building the search index in Part 2.